In [2]:
import pandas as pd

# 1. Configuração do Conjunto de Dados
dados = [
    ['infantil', 'miopia', 'não', 'reduzida', 'nenhuma'],
    ['infantil', 'miopia', 'sim', 'normal', 'gelatinosa'],
    ['infantil', 'hipermetropia', 'não', 'normal', 'gelatinosa'],
    ['infantil', 'hipermetropia', 'sim', 'normal', 'dura'],
    ['adolescente', 'miopia', 'não', 'reduzida', 'gelatinosa'],
    ['adolescente', 'miopia', 'sim', 'reduzida', 'nenhuma'],
    ['adolescente', 'miopia', 'não', 'normal', 'dura'],
    ['adolescente', 'hipermetropia', 'não', 'reduzida', 'gelatinosa'],
    ['adolescente', 'hipermetropia', 'sim', 'normal', 'dura'],
    ['adulto', 'miopia', 'não', 'normal', 'gelatinosa'],
    ['adulto', 'miopia', 'sim', 'normal', 'dura'],
    ['adulto', 'miopia', 'sim', 'normal', 'gelatinosa'],
    ['adulto', 'hipermetropia', 'não', 'reduzida', 'nenhuma'],
    ['adulto', 'hipermetropia', 'sim', 'normal', 'gelatinosa'],
    ['adulto', 'hipermetropia', 'não', 'normal', 'gelatinosa']
]

colunas = ['Idade', 'Diagnóstico', 'Astigmatismo', 'Taxa lacrimal', 'Tipo de lente']
df = pd.DataFrame(dados, columns=colunas)

In [4]:
# ---------------------------------------------------------
# EXERCÍCIO 1: Tabela de Contagens e Probabilidades (Laplace)
# ---------------------------------------------------------
print("--- EXERCÍCIO 1: CONTAGENS E PROBABILIDADES SUAVIZADAS (α=1) ---")
alpha = 1
alvo = 'Tipo de lente'
classes = df[alvo].unique()
features = [c for c in df.columns if c != alvo]

# Probabilidade a priori (P(Classe)) - Sem suavização (padrão usual)
p_classes = df[alvo].value_counts(normalize=True).to_dict()

tabelas_probs = {}

for feature in features:
    valores_unicos = df[feature].unique()
    k_valores = len(valores_unicos) # Quantidade de valores distintos da feature (|V|)

    linhas_tabela = []
    for c in classes:
        df_classe = df[df[alvo] == c]
        count_classe = len(df_classe)

        for v in valores_unicos:
            # Contagem real
            count_feature_classe = len(df_classe[df_classe[feature] == v])

            # Cálculo com Estimador Laplaciano
            # P(x|c) = (count(x, c) + alpha) / (count(c) + alpha * |V|)
            prob_suavizada = (count_feature_classe + alpha) / (count_classe + alpha * k_valores)

            linhas_tabela.append({
                'Feature': feature,
                'Valor': v,
                'Classe': c,
                'Contagem': count_feature_classe,
                'P(Valor|Classe)': prob_suavizada
            })

    tabelas_probs[feature] = pd.DataFrame(linhas_tabela)
    print(f"\nFeature: {feature}")
    print(tabelas_probs[feature].to_string(index=False))

--- EXERCÍCIO 1: CONTAGENS E PROBABILIDADES SUAVIZADAS (α=1) ---

Feature: Idade
Feature       Valor     Classe  Contagem  P(Valor|Classe)
  Idade    infantil    nenhuma         1         0.333333
  Idade adolescente    nenhuma         1         0.333333
  Idade      adulto    nenhuma         1         0.333333
  Idade    infantil gelatinosa         2         0.272727
  Idade adolescente gelatinosa         2         0.272727
  Idade      adulto gelatinosa         4         0.454545
  Idade    infantil       dura         1         0.285714
  Idade adolescente       dura         2         0.428571
  Idade      adulto       dura         1         0.285714

Feature: Diagnóstico
    Feature         Valor     Classe  Contagem  P(Valor|Classe)
Diagnóstico        miopia    nenhuma         2              0.6
Diagnóstico hipermetropia    nenhuma         1              0.4
Diagnóstico        miopia gelatinosa         4              0.5
Diagnóstico hipermetropia gelatinosa         4              0

In [6]:
# ---------------------------------------------------------
# EXERCÍCIO 2: Previsão para novo paciente
# ---------------------------------------------------------
print("\n\n--- EXERCÍCIO 2: PREVISÃO NAIVE BAYES ---")
paciente = {
    'Idade': 'infantil',
    'Diagnóstico': 'hipermetropia',
    'Astigmatismo': 'não',
    'Taxa lacrimal': 'reduzida'
}

scores = {}
print("Cálculo dos Scores brutos:")
for c in classes:
    # Score = P(Classe) * P(Idade|Classe) * P(Diag|Classe) * P(Astig|Classe) * P(Taxa|Classe)
    score_classe = p_classes[c]

    for feature, valor in paciente.items():
        # Busca a probabilidade suavizada calculada no passo 1
        df_feat = tabelas_probs[feature]
        prob = df_feat[(df_feat['Valor'] == valor) & (df_feat['Classe'] == c)]['P(Valor|Classe)'].values[0]
        score_classe *= prob

    scores[c] = score_classe
    print(f"Score não normalizado para '{c}': {score_classe:.6f}")

# Normalizando as probabilidades
soma_scores = sum(scores.values())
probs_normalizadas = {c: (score / soma_scores) for c, score in scores.items()}

print("\nProbabilidades Normalizadas:")
for c, prob in probs_normalizadas.items():
    print(f"P(Tipo de lente = {c} | Paciente) = {prob*100:.2f}%")

classe_prevista = max(probs_normalizadas, key=probs_normalizadas.get)
print(f"\n=> PREVISÃO FINAL: {classe_prevista.upper()}")



--- EXERCÍCIO 2: PREVISÃO NAIVE BAYES ---
Cálculo dos Scores brutos:
Score não normalizado para 'nenhuma': 0.012800
Score não normalizado para 'gelatinosa': 0.013091
Score não normalizado para 'dura': 0.002116

Probabilidades Normalizadas:
P(Tipo de lente = nenhuma | Paciente) = 45.70%
P(Tipo de lente = gelatinosa | Paciente) = 46.74%
P(Tipo de lente = dura | Paciente) = 7.56%

=> PREVISÃO FINAL: GELATINOSA
